In [ ]:
!pip uninstall -y datasets huggingface_hub fsspec
!pip install datasets==2.19.0 huggingface_hub==0.23.0 fsspec==2024.3.1
!pip install transformers==4.41.2 accelerate torch

Found existing installation: datasets 4.0.0
Uninstalling datasets-4.0.0:
  Successfully uninstalled datasets-4.0.0
Found existing installation: huggingface_hub 1.19.0
Uninstalling huggingface_hub-1.19.0:
  Successfully uninstalled huggingface_hub-1.19.0
Found existing installation: fsspec 2025.3.0
Uninstalling fsspec-2025.3.0:
  Successfully uninstalled fsspec-2025.3.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.2/401.2 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 14.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires huggingface-hub<2.0,>=0.33.5, but you have huggingface-hub 0.23.0 which is incompatible.
diffusers 0.38.0 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.23.0 which 

In [ ]:
import datasets
import transformers
import huggingface_hub

print("datasets:", datasets.__version__)
print("transformers:", transformers.__version__)
print("hub:", huggingface_hub.__version__)

datasets: 2.19.0
transformers: 4.41.2
hub: 0.23.0


In [ ]:
import pandas as pd

data = {
    "input": [
        "I am stressed about my exams",
        "I feel lonely",
        "I am very anxious",
        "I had a bad day",
        "I feel overwhelmed",
        "I am sad",
        "Nobody understands me",
        "I feel nervous about my future"
    ],
    "response": [
        "I am sorry you are feeling stressed. Try taking things one step at a time and remember that your effort matters.",
        "Feeling lonely can be difficult. Would you like to talk more about what is making you feel this way?",
        "Anxiety can feel overwhelming. Taking a few deep breaths may help you feel more grounded.",
        "I am sorry your day was difficult. Tomorrow is a new opportunity and your feelings are valid.",
        "It sounds like you have a lot on your mind. Consider taking a short break and focusing on one task at a time.",
        "I am sorry you are feeling sad. It can help to talk with someone you trust.",
        "That sounds painful. Your thoughts and feelings are important and deserve to be heard.",
        "It is normal to feel uncertain about the future. Try focusing on the things you can control today."
    ]
}

df = pd.DataFrame(data)
df.to_csv("mental_health_dataset.csv", index=False)

print(df.head())

                          input  \
0  I am stressed about my exams   
1                 I feel lonely   
2             I am very anxious   
3               I had a bad day   
4            I feel overwhelmed   

                                            response  
0  I am sorry you are feeling stressed. Try takin...  
1  Feeling lonely can be difficult. Would you lik...  
2  Anxiety can feel overwhelming. Taking a few de...  
3  I am sorry your day was difficult. Tomorrow is...  
4  It sounds like you have a lot on your mind. Co...  


In [ ]:
from datasets import Dataset
import pandas as pd

df = pd.read_csv("mental_health_dataset.csv")

dataset = Dataset.from_pandas(df)

print(dataset)

Dataset({
    features: ['input', 'response'],
    num_rows: 8
})


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token

print("Model Loaded")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model Loaded


In [ ]:
def preprocess(example):

    text = f"User: {example['input']}\nBot: {example['response']}"

    result = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=128
    )

    result["labels"] = result["input_ids"].copy()

    return result

tokenized_dataset = dataset.map(preprocess)

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./mental_health_bot",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    logging_steps=1
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

trainer.train()

RuntimeError: Failed to import transformers.trainer because of the following error (look up to see its traceback):
cannot import name 'LocalEntryNotFoundError' from 'huggingface_hub.errors' (/usr/local/lib/python3.12/dist-packages/huggingface_hub/errors.py)